# S26 — Efficiency and Scale

**Module 4**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s26_efficiency_and_scale.ipynb)

Every cell below is a worked example from the [S26 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s26/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s26.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s26.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## Number formats: what half precision actually buys and costs


*Expected output starts with:* `torch.float32    max 3.403e+38  smallest normal 1.175e-38  eps 1.192e-07`


In [ ]:
import torch

torch.manual_seed(0)

for dt in [torch.float32, torch.float16, torch.bfloat16]:
    fi = torch.finfo(dt)
    print(f"{str(dt):15}  max {fi.max:.3e}  smallest normal {fi.tiny:.3e}  eps {fi.eps:.3e}")

print()
# fp16 overflows where bf16 does not (but bf16 rounds coarsely)
x = torch.tensor(70000.0)
print(f"70000 in fp16: {x.to(torch.float16).item()}")     # overflow -> inf
print(f"70000 in bf16: {x.to(torch.bfloat16).item()}")    # fits, but rounded

# precision: 1 + 2^-10 is representable in fp16, lost in bf16
y = torch.tensor(1.0 + 2**-10)
print(f"1 + 2^-10 in fp32: {y.item():.10f}")
print(f"1 + 2^-10 in fp16: {y.to(torch.float16).item():.10f}")
print(f"1 + 2^-10 in bf16: {y.to(torch.bfloat16).item():.10f}")

print()
# the weight-update hazard: a small update vanishes when added to a large fp16 weight
w32 = torch.tensor(2048.0); w16 = w32.to(torch.float16)
print(f"fp32: 2048 + 1 = {(w32 + 1).item()}")
print(f"fp16: 2048 + 1 = {(w16 + torch.tensor(1.0, dtype=torch.float16)).item()}")

# gradient underflow: tiny fp16 gradients flush to zero -> loss scaling exists for this
g = torch.tensor(1e-8)
print(f"gradient 1e-8 in fp16: {g.to(torch.float16).item()}")
print(f"after 1024x loss scaling: {(g * 1024).to(torch.float16).item():.3e}")

## Where training memory actually goes


*Expected output starts with:* `model: 7B parameters, Adam, fp16/fp32 mixed precision`


In [ ]:
GB = 1024**3

def training_memory(n_params, n_gpus=1, zero_stage=0):
    """Bytes per GPU for mixed-precision Adam training (ZeRO-style sharding)."""
    weights_fp16 = 2 * n_params
    grads_fp16 = 2 * n_params
    # optimizer states kept in fp32: master weights + Adam m + Adam v
    opt_fp32 = (4 + 4 + 4) * n_params
    if zero_stage >= 1:
        opt_fp32 /= n_gpus
    if zero_stage >= 2:
        grads_fp16 /= n_gpus
    if zero_stage >= 3:
        weights_fp16 /= n_gpus
    return weights_fp16 + grads_fp16 + opt_fp32

N = 7_000_000_000
print(f"model: {N/1e9:.0f}B parameters, Adam, fp16/fp32 mixed precision")
print(f"inference only, fp16 weights:      {2*N/GB:6.1f} GB")
print(f"training, single GPU:              {training_memory(N)/GB:6.1f} GB")
for stage in [1, 2, 3]:
    m = training_memory(N, n_gpus=8, zero_stage=stage)
    print(f"training, 8 GPUs, ZeRO stage {stage}:    {m/GB:6.1f} GB per GPU")
print("(activations, comm buffers, and fragmentation come on top of this)")

## Quantization: inference in 8 bits (or fewer)


*Expected output starts with:* `well-behaved W, per-tensor :  max weight err 0.00090  relative matmul err 0.01032`


In [ ]:
import torch

torch.manual_seed(0)

def quantize_int8(w, per_channel=False):
    """Symmetric int8 quantization; returns dequantized weights."""
    if per_channel:
        scale = w.abs().max(dim=1, keepdim=True).values / 127
    else:
        scale = w.abs().max() / 127
    wq = (w / scale).round().clamp(-127, 127)
    return wq * scale

def report(name, w, x):
    y = w @ x
    for mode, pc in [("per-tensor", False), ("per-channel", True)]:
        wdq = quantize_int8(w, per_channel=pc)
        w_err = (wdq - w).abs().max()
        y_err = (wdq @ x - y).norm() / y.norm()
        print(f"{name}, {mode:11}:  max weight err {w_err.item():.5f}  "
              f"relative matmul err {y_err.item():.5f}")

w = torch.randn(256, 256) * 0.05          # typical trained-weight scale
x = torch.randn(256, 64)
report("well-behaved W", w, x)

w_out = w.clone()
w_out[0, 0] = 5.0                         # one outlier weight, 100x the rest
report("W with outlier", w_out, x)

## Pruning: removing weights outright


*Expected output starts with:* `trained baseline: 4352 prunable weights, test acc 0.9900`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Classification data: two interleaved moons
def moons(n, gen=None):
    t = torch.rand(n, generator=gen) * torch.pi
    x = torch.where(
        (torch.arange(n) % 2 == 0).unsqueeze(1),
        torch.stack([torch.cos(t), torch.sin(t)], dim=1),
        torch.stack([1 - torch.cos(t), 0.5 - torch.sin(t)], dim=1),
    ) + 0.15 * torch.randn(n, 2, generator=gen)
    y = (torch.arange(n) % 2).long()
    return x, y

g = torch.Generator().manual_seed(0)
x_tr, y_tr = moons(2000, g)
x_te, y_te = moons(1000, g)

net = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(),
                    nn.Linear(64, 2))
opt = torch.optim.Adam(net.parameters(), lr=1e-2)
for step in range(1500):
    idx = torch.randint(0, len(x_tr), (128,))
    loss = nn.functional.cross_entropy(net(x_tr[idx]), y_tr[idx])
    opt.zero_grad(); loss.backward(); opt.step()

def accuracy():
    with torch.no_grad():
        return (net(x_te).argmax(1) == y_te).float().mean().item()

weights = [m.weight for m in net.modules() if isinstance(m, nn.Linear)]
original = [w.detach().clone() for w in weights]
print(f"trained baseline: {sum(w.numel() for w in weights)} prunable weights, "
      f"test acc {accuracy():.4f}")

def prune_and_eval(sparsity, per_layer):
    with torch.no_grad():
        for w, orig in zip(weights, original):
            w.copy_(orig)
        if per_layer:
            masks = [(w.abs() > w.abs().flatten().quantile(sparsity)).float()
                     for w in weights]
        else:  # one global threshold across all layers
            th = torch.cat([w.abs().flatten() for w in weights]).quantile(sparsity)
            masks = [(w.abs() > th).float() for w in weights]
        for w, m in zip(weights, masks):
            w.mul_(m)
    acc = accuracy()
    out_kept = masks[-1].mean().item()   # surviving fraction in the output layer
    # brief fine-tune with the masks enforced after each step
    opt_ft = torch.optim.Adam(net.parameters(), lr=1e-3)
    for step in range(200):
        idx = torch.randint(0, len(x_tr), (128,))
        loss = nn.functional.cross_entropy(net(x_tr[idx]), y_tr[idx])
        opt_ft.zero_grad(); loss.backward(); opt_ft.step()
        with torch.no_grad():
            for w, m in zip(weights, masks):
                w.mul_(m)
    return acc, accuracy(), out_kept

print(f"{'sparsity':>9} {'scheme':>10} {'acc pruned':>11} {'after tune':>11} "
      f"{'output layer kept':>18}")
for sparsity in [0.5, 0.8, 0.9, 0.95]:
    for per_layer in [False, True]:
        acc, acc_ft, kept = prune_and_eval(sparsity, per_layer)
        scheme = "per-layer" if per_layer else "global"
        print(f"{sparsity:>9.0%} {scheme:>10} {acc:>11.4f} {acc_ft:>11.4f} {kept:>17.0%}")

## Serving transformers: the KV cache


*Expected output starts with:* `model                         KV/token   4k ctx  32k ctx  128k ctx`


In [ ]:
GB = 1024**3
MB = 1024**2

def kv_bytes_per_token(n_layers, n_kv_heads, head_dim, bytes_per_val=2):
    # K and V, per layer, per kv head, per head dimension
    return 2 * n_layers * n_kv_heads * head_dim * bytes_per_val

# Two real architectures (both ~7-8B parameters, fp16 weights ~13-15 GB):
# multi-head attention (every head stores K/V) vs grouped-query attention (8 shared KV heads)
configs = [
    ("7B, MHA (Llama-2-7B-like)", 32, 32, 128),
    ("8B, GQA (Llama-3-8B-like)", 32, 8, 128),
]

print(f"{'model':28} {'KV/token':>9} {'4k ctx':>8} {'32k ctx':>8} {'128k ctx':>9}")
for name, L, H, D in configs:
    per_tok = kv_bytes_per_token(L, H, D)
    row = [per_tok * n / GB for n in (4096, 32768, 131072)]
    print(f"{name:28} {per_tok/1024:>7.0f}KB {row[0]:>7.2f}G {row[1]:>7.2f}G {row[2]:>8.2f}G")

# Serving arithmetic: one 80 GB GPU, fp16 weights loaded, rest is KV budget
weights = {"7B, MHA (Llama-2-7B-like)": 2 * 7e9, "8B, GQA (Llama-3-8B-like)": 2 * 8e9}
print()
print("concurrent 8k-token sequences that fit in 80 GB alongside the weights:")
for name, L, H, D in configs:
    kv_per_seq = kv_bytes_per_token(L, H, D) * 8192
    budget = 80 * GB - weights[name]
    print(f"{name:28} {int(budget // kv_per_seq):>4d} sequences "
          f"({kv_per_seq/GB:.2f} GB per sequence)")

## Batching: throughput against latency


*Expected output starts with:* ` batch  step time  per-user tok/s  total tok/s  GPU-seconds/1M tok`


In [ ]:
GB = 1024**3

# A deliberately simple model of memory-bound decoding:
# each decode step must stream (weights + the batch's KV cache) from GPU memory.
# time per step ~= bytes moved / memory bandwidth.
weights_bytes = 2 * 8e9                     # 8B model, fp16
kv_per_seq = 131072 * 4096                  # 8k-token context, GQA (from the KV script)
bandwidth = 2.0e12                          # 2 TB/s HBM, an A100-80GB-class figure

print(f"{'batch':>6} {'step time':>10} {'per-user tok/s':>15} {'total tok/s':>12} "
      f"{'GPU-seconds/1M tok':>19}")
for batch in [1, 2, 4, 8, 16, 32, 64, 128, 256]:
    bytes_per_step = weights_bytes + batch * kv_per_seq
    step_time = bytes_per_step / bandwidth          # seconds per decode step
    per_user = 1 / step_time                        # each user gets 1 token per step
    total = batch / step_time
    cost = 1e6 / total
    print(f"{batch:>6} {step_time*1000:>8.2f}ms {per_user:>15.1f} {total:>12.0f} "
          f"{cost:>19.1f}")

## Try it yourself

1. Extend the quantization script to int4 (scale by `max|W| / 7`). How do the two matmul errors change? At what bit width does the well-behaved matrix's error exceed 5%?
2. Extend the memory script to a 70B model and per-GPU memory of 80 GB. How many GPUs does each ZeRO stage need just to hold the training state (ignoring activations)?
3. Simulate loss scaling end to end: generate 10,000 gradients from `N(0, 1e-7^2)`, cast to fp16 with and without a 1024x scale, and report what fraction survive as nonzero in each case.
4. Compute the memory arithmetic for LoRA fine-tuning of a 7B model with 0.5% trainable parameters: fp16 frozen weights plus full Adam state for the trainable part only. Compare with full fine-tuning.
5. In the pruning script, prune only the two hidden layers and leave the output layer untouched, then redo the global-pruning rows. How far does the 90% catastrophe move? What does this tell you about where the global scheme's failure actually lived?
6. Extend the batching simulation: hold the total memory at 80 GB and recompute the maximum feasible batch at context lengths 2k, 8k, 32k (batch limited by KV fitting in memory). Plot or tabulate achievable total throughput versus context length. Where does long context actually cost you?
7. Redo the KV arithmetic for a 70B model (80 layers, 8 KV heads, head_dim 128) at 128k context. Does a single 80 GB GPU still hold even one sequence alongside int4-quantized weights (about 35 GB)?


---

Full discussion of everything above: [S26 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s26/).
